Converts emotional manipulation corpus into ChunkedCorpus format. Merges specified labels into 'other' category.

In [1]:
%load_ext autoreload
%autoreload all

#import notebook helpers
from IPython.display import display,  clear_output
from tqdm import tqdm
import builtins

#import libraries
import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

#define base paths
origBasePath = os.path.abspath("./original/ktu")
print(origBasePath)

preprocBasePath = os.path.abspath("./preproc/ktu")
print(preprocBasePath)

/home/user/Work/misijos/detector-all/detector-emotmanip/data/original/ktu
/home/user/Work/misijos/detector-all/detector-emotmanip/data/preproc/ktu


In [2]:
#define file paths
inFilePath = os.path.join(origBasePath, "corpus_AICP_FIMI_2026-01-24.json")
print(inFilePath)

outFilePath = os.path.join(preprocBasePath, "corpus_AICP_FIMI_2026-01-24-merged-lbls.ranged.json")
print(outFilePath)

/home/user/Work/misijos/detector-all/detector-emotmanip/data/original/ktu/corpus_AICP_FIMI_2026-01-24.json
/home/user/Work/misijos/detector-all/detector-emotmanip/data/preproc/ktu/corpus_AICP_FIMI_2026-01-24-merged-lbls.ranged.json


In [3]:
#read the input

#define schema of the input
from pydantic import BaseModel, Field
from typing import List
import json

class InAnnot(BaseModel):
    technique : str
    fragment : str
    start : int
    end : int

class InText(BaseModel):
    comment_id : int
    content : str
    annotations : List[InAnnot] = Field(default=[])
    found_manipulation : str

#read the input file
inTexts : List[InText] = []

with open(inFilePath, mode="r", encoding="utf-8") as file:
    inJson = json.load(file)
    inTexts = [InText(**it) for it in inJson]

In [4]:
#build the label set

mergeLabelIdxs = [1, 3, 6, 7, 8, 10, 11, 13]

#get all labels
from typing import Set

labelSet : Set[str] = set()
for inText in inTexts:
    for annot in inText.annotations:
        labelSet.add(annot.technique.strip().lower())

#build label list
labelLst = list(labelSet)
labelLst.sort()

labelLst.insert(0, "none")    #neutral label

#remove merged labels and add 'other'
labelLst = [x for idx, x in enumerate(labelLst) if idx not in mergeLabelIdxs]
labelLst.insert(1, "other")

#build label dictionary
labelDict = { idx: lbl for idx, lbl in enumerate(labelLst) }
display(labelDict)

#build label groups
numHeadLabels = [len(labelDict)]
display(numHeadLabels)

{0: 'none',
 1: 'other',
 2: 'doubt',
 3: 'fearmongering',
 4: 'flag-waving',
 5: 'minimisation',
 6: 'scapegoating',
 7: 'support'}

[8]

In [5]:
#build chunked corpus

from corpus.chunkedCorpus import ChunkedCorpus, ChunkedText, Chunk, ChunkSplitter

#label to assign to neutral chunks
labelNone = 0
print(f"Neutral label is: {labelNone}")

labelOther = 1
print(f"Other label is: {labelOther}")

#
corpus = ChunkedCorpus(labelDefs = labelDict, labelGrps = numHeadLabels, texts = list())
for inTextIdx, inText in tqdm(enumerate(inTexts), "Input texts processed"):
    #skip empty texts, just in case
    if inText.content.strip() == 0:
        continue

    #
    chunks : List[Chunk] = []

    #initialize chunk splitter for building neutral chunks
    cs = ChunkSplitter(builtins.len(inText.content)-1)

    #build annotation chunks, split off from neutral
    for inAnnotIdx, inAnnot in enumerate(inText.annotations):
        #get chunk label index, use index of 'other' if not found in label dictionary
        labelIdxSel = [idx for idx, lbl in labelDict.items() if lbl == inAnnot.technique.strip().lower()]
        labelIdx = labelIdxSel[0] if len(labelIdxSel) > 0 else labelOther

        #build chunk for annotation
        chunkLen = inAnnot.end - inAnnot.start        
        chunk = Chunk(start=inAnnot.start, len=chunkLen, label=labelIdx)

        #register with chunk list
        chunks.append(chunk)

        #splitt off the annotated range from neutral ranges
        cs.takeOut(inAnnot.start, chunkLen)

    #build the neutral chunks and add to the chunk links
    neutralChunks = cs.getChunks(labelNone)
    chunks = chunks + neutralChunks

    #build the chunked text and register with the corpus
    chunkedText = ChunkedText(id=inTextIdx, text=inText.content, chunks = chunks)
    corpus.texts.append(chunkedText)

#save results
corpus.saveToJson(outFilePath)

Neutral label is: 0
Other label is: 1


Input texts processed: 7306it [00:00, 38349.78it/s]
